# Tokenizers: The Bridge Between English and Math

Neural networks, including Large Language Models (LLMs) and Diffusion models, cannot read English. They only understand numbers (tensors/matrices).

Before we can send a prompt to an AI model, we must translate our text into a mathematical format. A **Tokenizer** is the dictionary and the engine that performs this exact translation. It chops our text into pieces, looks up each piece in a massive dictionary, and replaces it with a unique integer ID.

## 1. The Core Problem

Historically, early AI researchers tried two different methods to translate text, both of which failed for massive models:

**Attempt A: Word-Level Tokenization**
* **How it works:** Assign a unique number to every single word in the English language.
* **The Problem:** The dictionary gets too big. What about typos? What about new words like "crypto" or "COVID"? If the model sees a word it doesn't have in its dictionary, it replaces it with a `<UNK>` (Unknown) token, instantly losing the meaning of the sentence.

**Attempt B: Character-Level Tokenization**
* **How it works:** Assign a unique number to the 26 letters of the alphabet, plus punctuation.
* **The Problem:** It loses semantic meaning. The letter "c" means nothing on its own. Furthermore, a single sentence becomes hundreds of numbers, destroying the model's Context Window limits and causing Out of Memory (OOM) errors.

## 2. Subword Tokenization (The Modern Standard)

Modern models (like GPT-4, Llama 3, and BERT) use **Subword Tokenization**. This is the "Goldilocks" solution.

Instead of chopping by whole words or individual letters, it chops text by *common syllables and word fragments*.

**How it works (Byte-Pair Encoding / WordPiece):**
It breaks down complex or unknown words into smaller, known logical chunks.
* Example: The word `"unbelievably"` might not be in the dictionary.
* The tokenizer splits it into: `["un", "##believ", "##ably"]`
* *Note: The `##` (or sometimes a `Ġ` symbol) just tells the model that this piece attaches directly to the piece before it without a space.*

**The Result:** The model never gets an `<UNK>` (Unknown) error! If it sees a completely made-up word, it will just chop it all the way down to its base letters if it has to. It forces the model to understand the *roots* of words, making it incredibly smart at deducing the meaning of new vocabulary.

## 3. The Hugging Face Tokenization Pipeline

When you call a Hugging Face `AutoTokenizer`, it secretly runs your text through four distinct steps:

1. **Pre-processing / Normalization:** It cleans the text (e.g., removing extra spaces, standardizing accents, sometimes lowercasing).
2. **Pre-tokenization:** It splits the sentence into raw words based on spaces and punctuation.
3. **Tokenization:** It applies the Subword algorithm (BPE/WordPiece) to chop the words into the final subword chunks.
4. **Post-processing:** This is crucial! It adds **Special Tokens** that the specific neural network requires to understand where the prompt starts and ends.
   * Example: BERT requires a `[CLS]` (Classification) token at the start and a `[SEP]` (Separator) token at the end.
   * Example: Llama requires a `<|begin_of_text|>` token.

In [2]:
# Imports:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoTokenizer

In [3]:
# Connecting to Hugging Face:
load_dotenv(override= True)
hf_token = os.getenv('HF_TOKEN')

if hf_token and hf_token.startswith('hf_'):
    print('Hugging Face Token Found')
    login(token= hf_token, add_to_git_credential= True)
    print('Successfully logged into Hugging Face')
else:
    print('Hugging Face Token not found')

Hugging Face Token Found


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Successfully logged into Hugging Face


In [4]:
# Checking GPU Access:

import torch

gpu_availability = torch.cuda.is_available()
if gpu_availability:
    print('GPU Available')
    device = torch.cuda.get_device_name()
    print(device)
else:
    print('GPU not available')

GPU Available
NVIDIA GeForce RTX 4080 Laptop GPU
